# DiffPO Confounding Diffusion Analysis

Compares DDIM denoising trajectories between DiffPO trained on clean IHDP (`naive_full`) and
DiffPO trained on confounded IHDP (`naive_conf`).

In [18]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml
from scipy.stats import gaussian_kde

from src.config import Config
from src.data import load_ihdp, make_ihdp_confounded
from src.model import DiffPO, _DiffusionBase

In [2]:
CLEAN_CHECKPOINT = "checkpoints/final_model_naive_full_2026-08-03T16_21_12.pth"
CONF_CHECKPOINT = "checkpoints/final_model_naive_conf_2026-08-21T12_31_09_rep1.pth"
N_PER_GROUP = 10  # subjects per momblack group
SEED = 0
DEVICE = torch.device("cpu")

In [ ]:
with open("config/ihdp.yaml") as f:
    cfg = Config.model_validate(yaml.safe_load(f))

clean_model = DiffPO(cfg.vae, cfg.diffusion)
clean_model.load_state_dict(torch.load(CLEAN_CHECKPOINT, map_location=DEVICE))
clean_model.eval()

conf_model = DiffPO(cfg.vae, cfg.diffusion)
conf_model.load_state_dict(torch.load(CONF_CHECKPOINT, map_location=DEVICE))
conf_model.eval()

print("clean_model.L:", clean_model.L, "  conf_model.L:", conf_model.L)

clean_model.L: 200   conf_model.L: 200


In [5]:
train_ds_clean, val_ds_clean, test_ds_clean, y_std = load_ihdp(
    cfg.data.path,
    replication=cfg.data.replication,
    train_ratio=cfg.data.train_ratio,
    test_ratio=cfg.data.test_ratio,
)
train_ds_conf, val_ds_conf, test_ds_conf = (
    make_ihdp_confounded(ds, effect=cfg.data.confounder_effect)
    for ds in (train_ds_clean, val_ds_clean, test_ds_clean)
)


def _clip_value(train_ds):
    if not cfg.diffusion.clip_denoised:
        return None
    y_both = _DiffusionBase._assemble_yboth(train_ds.a, train_ds.y, train_ds.y_cf)
    return 2 * y_both.abs().max().item()


clip_value_clean = _clip_value(train_ds_clean)
clip_value_conf = _clip_value(train_ds_conf)
print("clip_value_clean:", clip_value_clean, "  clip_value_conf:", clip_value_conf)

clip_value_clean: 6.018764972686768   clip_value_conf: 7.191953659057617


In [ ]:
rng = np.random.default_rng(SEED)
confounder = test_ds_clean.confounder
assert confounder is not None

idx_flipped = np.flatnonzero(confounder == 1)
idx_control = np.flatnonzero(confounder == 0)
assert len(idx_flipped) >= N_PER_GROUP, f"only {len(idx_flipped)} momblack==1 test subjects"
assert len(idx_control) >= N_PER_GROUP, f"only {len(idx_control)} momblack==0 test subjects"

subset_idx = np.concatenate(
    [
        rng.choice(idx_flipped, size=N_PER_GROUP, replace=False),
        rng.choice(idx_control, size=N_PER_GROUP, replace=False),
    ]
)
momblack_t = torch.as_tensor(confounder[subset_idx])
flipped_mask = momblack_t == 1
control_mask = momblack_t == 0

x = test_ds_clean.x[subset_idx]
a_clean = test_ds_clean.a[subset_idx]
a_conf = test_ds_conf.a[subset_idx]

mu0_clean = test_ds_clean.mu0[subset_idx]
mu1_clean = test_ds_clean.mu1[subset_idx]
mu0_conf = test_ds_conf.mu0[subset_idx]
mu1_conf = test_ds_conf.mu1[subset_idx]

# momblack==0 subjects must see identical (x, a) in both conditions (built-in control group);
# momblack==1 subjects must see exactly-flipped a.
assert torch.equal(a_clean[control_mask], a_conf[control_mask])
assert torch.equal(a_conf[flipped_mask], 1.0 - a_clean[flipped_mask])
assert torch.equal(test_ds_clean.x[subset_idx], test_ds_conf.x[subset_idx])

# mu0/mu1 must also agree on the control group. Using allclose as `make_ihdp_confounded`
# round-trips mu0/mu1 through denorm()/renorm() even for momblack==0 subjects (where the
# shift itself is a mathematical no-op). That float32 round trip is not bit-exact
assert torch.allclose(mu0_clean[control_mask], mu0_conf[control_mask], atol=1e-6)
assert torch.allclose(mu1_clean[control_mask], mu1_conf[control_mask], atol=1e-6)

print(
    f"Subset: {len(subset_idx)} subjects "
    f"({int(flipped_mask.sum())} momblack==1, {int(control_mask.sum())} momblack==0)"
)

print()

Subset: 20 subjects (10 momblack==1, 10 momblack==0)


In [23]:
print("Clean dataset cross tabulation")
print(
    pd.crosstab(
        test_ds_clean.confounder[subset_idx],
        test_ds_clean.a[subset_idx],
        rownames=["momblack"],
        colnames=["treatment"],
    )
)
print()

print("Confounded dataset cross tabulation:")
print(
    pd.crosstab(
        test_ds_conf.confounder[subset_idx],
        test_ds_conf.a[subset_idx],
        rownames=["momblack"],
        colnames=["treatment"],
    )
)

Clean dataset cross tabulation
treatment  0.0  1.0
momblack           
0.0          6    4
1.0          3    7

Confounded dataset cross tabulation:
treatment  0.0  1.0
momblack           
0.0          6    4
1.0          7    3
